TEST DIFFERENT LLM models 

In [ ]:
import json
from openai import OpenAI
import os
import faiss
import numpy as np
import matplotlib.pyplot as plt
from datetime import datetime
from pathlib import Path

# === Configuration ===
MODEL = "gpt-4.1"
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))
PHASES = ["take_off", "cruising", "landing"]

# === Prompt de résumé phase (tu l’utilises déjà) ===
PROMPT_PHASE_TEMPLATE = """
You are given a segment of flight data corresponding to the {phase_name} phase of a flight. The data is structured as a list of datapoints, 
each containing the aircraft’s state and environmental conditions at a given time.

Each datapoint is a dictionary with keys such as:
- lat, lon, track, altitude_ft, gspeed (knots), vspeed (ft/min), minutes_since_start (float)
- edr_300hPa, ellrod_300hPa, richardson_number_300hPa, cape,
- wind_speed_10000m, wind_dir_10000m, wind_speed_2m, wind_dir_2m.

Your task is to write a **continuous, expert-level narrative** describing this phase of flight, 
progressing through the datapoints chronologically.

For each datapoint:
1. Use the field `minutes_since_start` to reference time relative to takeoff.
   Example: “5 minutes after departure” (for takeoff) or “5 minutes into the flight” (for cruise/landing).
2. Describe the aircraft's position (lat/lon) and motion (heading, altitude, ground speed, vertical speed).
3. Include meteorological context: turbulence (EDR, Ellrod), stability (Richardson number), CAPE, and wind information.
4. Mention relevant geographic regions based on coordinates (e.g., cities, monuments, regions, landscapes, montains, seas, rivers etc...).
5. Use **smooth transitions** between datapoints:
   - “Then the aircraft turns east by 20° and climbs another 300 ft...”
   - “A few minutes later, it begins a gentle descent...”
   - “As it approaches the Tyrrhenian coast...”
6. Do not forget to include the plane's type at the begining of your description: {type}

Write the result as a natural, continuous, and professional description, not as bullet points. I want to description to be very detailed.

Here is the flight phase's record:
{phase_data}

Return the flights summary without any metatext"""

# === Génère le résumé via OpenAI pour n points de données ===
def generate_summary(phase_name, phase_data, aircraft_type):
    prompt = PROMPT_PHASE_TEMPLATE.format(
        phase_name=phase_name,
        phase_data=json.dumps(phase_data),
        type=aircraft_type
    )
    response = client.chat.completions.create(
        model="gpt-4.1",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.3
    )
    parsed = response.choices[0].message.content.strip()
    return parsed # narrative only



# === Construit le prompt final de prédiction ===
def build_prediction_prompt(target_summary, similar_summaries, N):
    prompt = f"""The target flight entered a GPS spoofed zone. Here is its trajectory summary before spoofing:
{target_summary}

Here are the summaries of the 5 most similar flights:"""
    for i, summ in enumerate(similar_summaries):
        prompt += f"\n\nSimilar flight {i+1}:\n{summ}"

    prompt += f"""

Based on these flights, where will the target flight be {N} minutes after the start of the phase?

⚠️ Return only the predicted position in this exact format:
LATITUDE, LONGITUDE
"""
    return prompt

# === Appel LLM pour la prédiction ===
def predict_position(prompt, model: str = MODEL):
    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
    return response.choices[0].message.content.strip()

In [ ]:
import os
import json
import faiss
from openai import OpenAI
from dotenv import load_dotenv
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY_2"))

# --------------------------
# CONFIGURATION
# --------------------------
EMBED_MODEL = "text-embedding-3-large"
K = 5  # number of similar flights retrieved
PHASE = "take_off"  # phase we test
EMBEDDINGS_PATH = "../../data/CDG-FCO/EMBEDDINGS"

# --------------------------
# LOAD DATA
# --------------------------
with open("../../data/CDG-FCO/flight_data_with_minutes_since_start.json", "r", encoding="utf-8") as f:
    flights_data = json.load(f)

with open("../../data/CDG-FCO/flight_summary_separated_into_phases.json", "r", encoding="utf-8") as f:
    summaries = json.load(f)

data_by_id = {f["flight_metadata"]["fr24_id"]: f for f in flights_data}
summary_by_phase = {
    s["fr24_id"]: {p: s.get(p, None) for p in ["take_off", "cruising", "landing"]}
    for s in summaries
}

print(f"✅ Loaded {len(flights_data)} flights with data and summaries.\n")


def save_phase_index(phase, index, ids, embeddings):
    faiss.write_index(index, f"{EMBEDDINGS_PATH}/faiss_index_{phase}.index")
    np.save(f"{EMBEDDINGS_PATH}/embeddings_{phase}.npy", embeddings)
    with open(f"{EMBEDDINGS_PATH}/ids_{phase}.json", "w") as f:
        json.dump(ids, f)
    print(f"💾 Saved FAISS index, embeddings and IDs for phase '{phase}'")


def load_phase_index(phase):
    index = faiss.read_index(f"{EMBEDDINGS_PATH}/faiss_index_{phase}.index")
    embeddings = np.load(f"{EMBEDDINGS_PATH}/embeddings_{phase}.npy")
    with open(f"{EMBEDDINGS_PATH}/ids_{phase}.json", "r") as f:
        ids = json.load(f)
    print(f"📂 Loaded FAISS index, embeddings and IDs for phase '{phase}'")
    return index, ids, embeddings

# --------------------------
# EMBEDDING HELPER
# --------------------------
def get_embeddings(texts, model=EMBED_MODEL, batch_size=32):
    """Embed multiple texts with normalization."""
    vecs = []
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        resp = client.embeddings.create(model=model, input=batch)
        embs = [np.array(e.embedding, dtype="float32") for e in resp.data]
        vecs.extend(embs)
    X = np.vstack(vecs)
    X /= np.linalg.norm(X, axis=1, keepdims=True)
    return X

# --------------------------
# BUILD RAG INDEX FOR EACH PHASE
# --------------------------

def build_phase_index(phase, force_rebuild=False):
    # 1) If files exist → load them
    if (not force_rebuild and 
        os.path.exists(f"{EMBEDDINGS_PATH}/faiss_index_{phase}.index") and
        os.path.exists(f"{EMBEDDINGS_PATH}/embeddings_{phase}.npy") and
        os.path.exists(f"{EMBEDDINGS_PATH}/ids_{phase}.json")):

        return load_phase_index(phase)

    # 2) Else → rebuild the index
    valid = [fid for fid, s in summary_by_phase.items() if s.get(phase)]
    texts = [summary_by_phase[fid][phase] for fid in valid]

    if not texts:
        raise ValueError(f"No valid summaries for phase '{phase}'")

    E = get_embeddings(texts)
    d = E.shape[1]

    index = faiss.IndexFlatIP(d)
    index.add(E)

    print(f"🔧 Index built for phase '{phase}' with {len(valid)} flights")

    # 3) Save everything for next time
    save_phase_index(phase, index, valid, E)

    return index, valid, E

In [ ]:
phase_index_take_off, phase_ids_take_off, _ = build_phase_index("take_off")
phase_index_cruising, phase_ids_cruising, _ = build_phase_index("cruising")
phase_index_landing, phase_ids_landing, _ = build_phase_index("landing")

In [ ]:
index_dictionary = {"take_off": phase_index_take_off, "cruising": phase_index_cruising, "landing": phase_index_landing}
ids_dictionary = {"take_off": phase_ids_take_off, "cruising": phase_ids_cruising, "landing": phase_ids_landing}

In [ ]:
import re

# === JSON cleanup ===
def clean_answer(raw_response: str):
    """
    Cleans and safely parses LLM JSON output (even if wrapped in markdown fences).
    """
    # Remove Markdown code fences (```json ... ```)
    cleaned = re.sub(r"^```(?:json|python)?", "", raw_response.strip(), flags=re.IGNORECASE)
    cleaned = re.sub(r"```$", "", cleaned.strip())

    # Remove common markdown artifacts
    cleaned = cleaned.replace("\n", "\\n").replace("\r", "\\r")

    # Some models may prepend language tag or commentary
    cleaned = re.sub(r"^json", "", cleaned.strip(), flags=re.IGNORECASE)

    # Extra cleanup for accidental trailing commas
    cleaned = re.sub(r",\s*([\]\}])", r"\1", cleaned)

    # Try JSON parsing
    try:
        return json.loads(cleaned)
    except json.JSONDecodeError as e:
        # print(f"⚠️ JSON parsing failed, attempting manual recovery: {e}")
        # # Attempt a secondary cleaning
        cleaned2 = cleaned.replace('\\n', ' ').replace('\\"', '"')
        return json.loads(cleaned2)

In [ ]:
PREDICT_POSITION_PROMPT = f"""
A flight entered a GPS spoofing zone. Therefore his position is now unknown and your task is to help the pilot 
by predicting the flight's latitude and longitude.
Here is its known trajectory (before spoofing):

{{partial_summary}}

The following are summaries of {{number_similar_flight}} similar flights for the same phase:

{{sim_text}}

{{predictions_history}}

Based on these, predict the geographic position of the target flight {{predict_minutes}} minutes after the beginning of the phase.
Return your answer as JSON: {{{{"lat": "the predicted latitude as a float", "lon": "the predicted longitude as a float"}}}} only.
"""

# --------------------------
# PREDICT POSITION VIA LLM
# --------------------------
def predict_position_from_prompt(partial_summary, similar_summaries, predict_minutes, prediction_history: list=[], model: str = MODEL):
    """Ask GPT to predict target position after N minutes using similar flights."""
    sim_text = "\n\n".join([f"Similar flight {i+1}:\n{txt}" for i, txt in enumerate(similar_summaries)])
    if not prediction_history:
        predictions_history = ""
    else:
        predictions_history = f"""Here are the predictions you already made. They are given as a list of triplet [latitude, longitude, number of minutes into the phase]. Use these past predictions to ensure that your next prediction is consistent.
Make sure the plane continues moving forward along its trajectory, does not remain at the same coordinates repeatedly, and does not move backward in a way that violates realistic flight dynamics.

PREDICTION HISTORY: {{prediction_history}}"""
        predictions_history = predictions_history.format(prediction_history=prediction_history)

    prompt = PREDICT_POSITION_PROMPT.format(
        partial_summary=partial_summary, 
        number_similar_flight=len(similar_summaries),
        sim_text=sim_text,
        predict_minutes=predict_minutes,
        predictions_history=predictions_history,
    )

    response = client.chat.completions.create(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        temperature=0
    )
        
    return clean_answer(response.choices[0].message.content), response.usage.total_tokens


In [ ]:
# --------------------------
# FIND SIMILAR FLIGHTS
# --------------------------
def find_similar_flights(partial_summary, target_id, phase, k=K, batch_size=None):
    """Embed the target partial summary and find K most similar full summaries."""
    q_vec = get_embeddings([partial_summary])
    sims, idxs = index_dictionary[phase].search(q_vec, k)
    ids_for_phase = ids_dictionary[phase]
    sims, idxs = sims[0], idxs[0]
    neighbors = [(ids_for_phase[i], float(sims[i])) for i in range(len(idxs)) if ids_for_phase[i] != target_id]
    return neighbors


In [ ]:
import matplotlib.pyplot as plt
import math

# ================================================
# CONFIGURATION EXPERIMENTALE
# ================================================
K = 5        # number of similar flights to retrieve

def get_mean_similar_flights_position(similar_ids, data_by_id, phase, t_min):
    """
    Computes the mean latitude and longitude of the similar flights at the point
    closest to t_min in terms of minutes_since_start.
    """

    latitudes = []
    longitudes = []

    for fid in similar_ids:
        flight = data_by_id.get(fid)
        if not flight:
            continue

        phase_data = flight.get(phase)
        if not phase_data or len(phase_data) == 0:
            continue

        # Find the point in the similar flight closest to t_min
        closest_point = min(
            phase_data,
            key=lambda p: abs(p["minutes_since_start"] - t_min)
        )

        lat = closest_point.get("lat", None)
        lon = closest_point.get("lon", None)

        if lat is not None and lon is not None:
            latitudes.append(lat)
            longitudes.append(lon)

    # If no valid positions were found
    if len(latitudes) == 0:
        return (None, None)

    # Compute mean
    mean_lat = sum(latitudes) / len(latitudes)
    mean_lon = sum(longitudes) / len(longitudes)

    return [mean_lat, mean_lon, t_min]

def haversine(lat1, lon1, lat2, lon2):
    R = 6371e3
    phi1, phi2 = math.radians(lat1), math.radians(lat2)
    dphi = math.radians(lat2 - lat1)
    dlambda = math.radians(lon2 - lon1)
    a = math.sin(dphi/2)**2 + math.cos(phi1)*math.cos(phi2)*math.sin(dlambda/2)**2
    return R * 2 * math.atan2(math.sqrt(a), math.sqrt(1-a))

def predict_flight(flight_id, phase, with_feedback_loop=True, k=K, batch_size=None, model: str = MODEL):
    # --------------------------
    # 1️⃣ Extraire les données du vol cible
    # --------------------------
    target_flight = data_by_id[flight_id]
    phase_data = target_flight[phase]
    aircraft_type = target_flight["flight_metadata"]["type"]

    # --------------------------
    # 2️⃣ Définir la zone de spoofing
    # --------------------------
    N = len(phase_data)
    SPOOF_INDEX = N // 2   # floor division → produit exactement la séparation souhaitée

    # print(f"📏 Phase length = {N} points — Observed = 0:{SPOOF_INDEX-1}, Predicted = {SPOOF_INDEX}:{N-1}")

    observed_data = phase_data[:SPOOF_INDEX]
    spoofed_data  = phase_data[SPOOF_INDEX:]

    # --------------------------
    # 3️⃣ Générer le résumé partiel (RAG query)
    # --------------------------
    partial_summary = generate_summary(phase, observed_data, aircraft_type)
    # print("🧩 Résumé partiel généré.\n")

    # --------------------------
    # 5️⃣ Prédire tous les points de la zone spoofée
    # --------------------------
    predicted_positions = []
    ground_truth_positions = []
    mean_similar_flights_position = []

    for i, point in enumerate(spoofed_data):
        t_min = point["minutes_since_start"]
        barometric_altitude = point["altitude_ft"]

        # --------------------------
        # 4️⃣ Trouver les vols similaires en prenant en compte l'altitude connu
        # --------------------------
        altitude_prompt = f"""{t_min} minutes into the {phase} phase, the aircraft reached an altitude of {barometric_altitude} feet."""
        partial_summary = f"""{partial_summary} {altitude_prompt}"""
        neighbors = find_similar_flights(partial_summary, target_id=flight_id, phase=phase, k=k, batch_size=batch_size)
        similar_ids = [fid for fid, sim in neighbors]
        similar_summaries = [summary_by_phase[fid][phase] for fid in similar_ids]

        # print(f"🔍 {len(similar_ids)} vols similaires trouvés :", similar_ids, "\n")

        # print(f"⏳ Prédiction point {SPOOF_INDEX+i} (t = {t_min} min)")

        if with_feedback_loop:
            prediction, num_token = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min,
                prediction_history=predicted_positions,
                model=model
            )
        else:
            prediction, num_token = predict_position_from_prompt(
                partial_summary,
                similar_summaries,
                predict_minutes=t_min,
                model=model
            )

        if prediction:
            predicted_positions.append([float(prediction["lat"]), float(prediction["lon"]), t_min]) # 

        ground_truth_positions.append([point["lat"], point["lon"], t_min])
        mean_similar_flights_position.append(get_mean_similar_flights_position(similar_ids, data_by_id, phase, t_min))

    mean_error = 0
    mean_similar_flight_error = 0

    for i in range(len(predicted_positions)):
        predicted_point = predicted_positions[i]
        lat1 = predicted_point[0]
        lon1 = predicted_point[1]
        ground_truth_point = ground_truth_positions[i]
        lat2 = ground_truth_point[0]
        lon2 = ground_truth_point[1]
        error = haversine(lat1, lon1, lat2, lon2)
        mean_error += error

        lat1 = mean_similar_flights_position[i][0]
        lon1 = mean_similar_flights_position[i][1]
        error = haversine(lat1, lon1, lat2, lon2)
        mean_similar_flight_error += error

    mean_error = mean_error / len(predicted_positions)
    mean_similar_flight_error = mean_similar_flight_error / len(mean_similar_flights_position)
    # print(f"📌 Avg Error = {mean_error:.2f} meters")

    return observed_data, ground_truth_positions, predicted_positions, round(mean_error/1000, 1), mean_similar_flights_position, round(mean_similar_flight_error/1000, 1), num_token

def plot_results(
        flight_id,
        phase,
        ground_truth_positions,
        predicted_positions,
        observed_data,
        mean_similar_flights_position
    ):
    plt.figure(figsize=(12, 10))

    # --------------------------------
    # Build timestamps
    # --------------------------------
    observed_times = [p["minutes_since_start"] for p in observed_data]

    # --------------------------------
    # Observed trajectory (GREEN)
    # --------------------------------
    obs_lats = [p["lat"] for p in observed_data]
    obs_lons = [p["lon"] for p in observed_data]

    plt.plot(obs_lons, obs_lats, color="green", linewidth=2, label="Observed (before spoofing)")
    plt.scatter(obs_lons, obs_lats, color="green", s=40)

    for lon, lat, t in zip(obs_lons, obs_lats, observed_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="green")

    # --------------------------------
    # Ground truth trajectory (BLUE)
    # --------------------------------
    gt_lats = [p[0] for p in ground_truth_positions]
    gt_lons = [p[1] for p in ground_truth_positions]
    ground_truth_times = [p[2] for p in ground_truth_positions]

    plt.plot(gt_lons, gt_lats, color="blue", linewidth=2, label="Ground Truth (spoofed region)")
    plt.scatter(gt_lons, gt_lats, color="blue", s=40)

    for lon, lat, t in zip(gt_lons, gt_lats, ground_truth_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="blue")

    # --------------------------------
    # Predicted trajectory (RED)
    # --------------------------------
    pred_lats = [p[0] for p in predicted_positions]
    pred_lons = [p[1] for p in predicted_positions]
    spoof_times = [p[2] for p in predicted_positions]

    plt.plot(pred_lons, pred_lats, color="red", linewidth=2, label="Predicted trajectory (LLM)")
    plt.scatter(pred_lons, pred_lats, color="red", s=40)

    for lon, lat, t in zip(pred_lons, pred_lats, spoof_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="red")

    # --------------------------------
    # Mean similar flights trajectory (ORANGE)
    # --------------------------------
    mean_lats = [p[0] for p in mean_similar_flights_position]
    mean_lons = [p[1] for p in mean_similar_flights_position]
    spoof_times = [p[2] for p in predicted_positions]

    plt.plot(mean_lons, mean_lats, color="orange", linewidth=2, linestyle="--",
             label="Mean similar flights trajectory")
    plt.scatter(mean_lons, mean_lats, color="orange", s=40)

    for lon, lat, t in zip(mean_lons, mean_lats, spoof_times):
        plt.text(lon, lat, f"{t:.1f}", fontsize=8, color="orange")

    # --------------------------------
    # Final layout
    # --------------------------------
    plt.title(
        f"Flight {flight_id} – Phase {phase}\n"
        f"Observed vs Ground Truth vs LLM Prediction",
        fontsize=14
    )
    plt.xlabel("Longitude")
    plt.ylabel("Latitude")
    plt.legend()
    plt.grid(True)
    plt.tight_layout()
    plt.show()


In [ ]:
# ================================================
# Lancer l'expérience pour les 3 vols
# ================================================
results = {
    "take_off": {"observed": [], "ground_truth": [], "LLMs": {}}, 
    "cruising": {"observed": [], "ground_truth": [], "LLMs": {}}, 
    "landing": {"observed": [], "ground_truth": [], "LLMs": {}}
    }

FLIGHT_ID = "39ba2570"

MODELS = [
    "gpt-3.5-turbo",
    "gpt-4",
    "gpt-4.1",
    "gpt-4-turbo",
    "gpt-4o",
    "gpt-4o-mini",
    "gpt-5.1",
    "gpt-5.1-instant",
    "gpt-5.2",
    "gpt-5.2-instant"
]



for phase in PHASES:
    observed_saved = False
    ground_truth_saved = False
    print(phase)
    for model in MODELS:
        print(model)
        try:
            observed, gt, pred, error, _, _, _ = predict_flight(FLIGHT_ID, phase, with_feedback_loop=True, model=model)
            if not observed_saved:
                results[phase]["observed"] = observed
                observed_saved = True
            if not ground_truth_saved:
                results[phase]["ground_truth"] = gt
                ground_truth_saved = True
            results[phase]["LLMs"][model] = {"predictions": pred, "mean_error": error}
        except:
            print(f"{model} failed")

results

In [ ]:
def get_ground_truth_and_observed(flight_id, phase):
    # --------------------------
    # 1️⃣ Extraire les données du vol cible
    # --------------------------
    target_flight = data_by_id[flight_id]
    phase_data = target_flight[phase]
    # --------------------------
    # 2️⃣ Définir la zone de spoofing
    # --------------------------
    N = len(phase_data)
    SPOOF_INDEX = N // 2   # floor division → produit exactement la séparation souhaitée

    # print(f"📏 Phase length = {N} points — Observed = 0:{SPOOF_INDEX-1}, Predicted = {SPOOF_INDEX}:{N-1}")

    observed_data = phase_data[:SPOOF_INDEX]
    spoofed_data  = phase_data[SPOOF_INDEX:]
    ground_truth_positions = []

    for i, point in enumerate(spoofed_data):
        t_min = point["minutes_since_start"]
        ground_truth_positions.append([point["lat"], point["lon"], t_min])

    observed_data = [[point["lat"], point["lon"], point["minutes_since_start"]] for point in observed_data]

    return observed_data, ground_truth_positions


In [ ]:
for phase in PHASES:
    observed, ground_truth = get_ground_truth_and_observed(FLIGHT_ID, phase)
    results[phase]["observed"] = observed
    results[phase]["ground_truth"] = ground_truth

In [ ]:
import json


OUTPUT_PATH = "LLM_COMPARISON.json"

with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

print(f"Saved to {OUTPUT_PATH}")


In [ ]:
MODELS_TO_COMPARE = [
    "gpt-4.1",
    "gpt-3.5-turbo",
    "gpt-4",
    "gpt-4o",
    "gpt-5.1",
    "gpt-5.2"
]


In [ ]:
import json
import matplotlib.pyplot as plt

# =========================
# Load data
# =========================
JSON_PATH = "LLM_COMPARISON.json"

with open(JSON_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

PHASES = ["take_off", "cruising", "landing"]


# =========================
# Plot
# =========================
for phase in PHASES:
    phase_data = data.get(phase, {})
    gt = phase_data.get("ground_truth", [])
    llms = phase_data.get("LLMs", {})

    if len(gt) == 0:
        print(f"Skipping {phase}: no ground truth")
        continue

    gt_lat = [p[0] for p in gt]
    gt_lon = [p[1] for p in gt]

    for model in MODELS_TO_COMPARE:
        if model not in llms:
            continue

        preds = llms[model]["predictions"]
        lat = [p[0] for p in preds]
        lon = [p[1] for p in preds]

        plt.figure(figsize=(7, 7))

        # ---- Ground truth (always dominant)
        plt.plot(
            gt_lon,
            gt_lat,
            linewidth=4,
            marker="o",
            label="Ground Truth",
            zorder=3
        )

        # ---- LLM prediction
        plt.plot(
            lon,
            lat,
            linewidth=3,
            linestyle="--",
            marker="o",
            label=model,
            zorder=2
        )

        # ---- Formatting
        plt.title(f"{phase} – {model}")
        plt.xlabel("Longitude")
        plt.ylabel("Latitude")
        plt.legend()
        plt.grid(True)
        plt.axis("equal")

        plt.show()


In [ ]:
import json
import math
import numpy as np

# ==========================================================
# FILE PATH
# ==========================================================
FILE_PATH = "LLM_comparison.json"

# ==========================================================
# OFFICIAL INPUT COST ($ per 1M tokens)
# ==========================================================
MODEL_INPUT_COST = {
    "gpt-3.5-turbo": 0.50,
    "gpt-4.1": 2.00,
    "gpt-4o": 2.50,
    "gpt-4o-mini": 0.15,
    "gpt-5.1": 1.25,
    "gpt-5.2": 1.75,
}

# ==========================================================
# HAVERSINE FUNCTION (km)
# ==========================================================
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])

    dlat = lat2 - lat1
    dlon = lon2 - lon1

    a = math.sin(dlat / 2)**2 + math.cos(lat1) * math.cos(lat2) * math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))

    return R * c


# ==========================================================
# LOAD JSON
# ==========================================================
with open(FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

results = {}

# ==========================================================
# COMPUTE METRICS
# ==========================================================
for phase, phase_data in data.items():

    ground_truth = phase_data["ground_truth"]
    models = phase_data["LLMs"]

    results[phase] = {}

    for model_name, model_data in models.items():

        predictions = model_data["predictions"]
        errors = []

        for gt, pred in zip(ground_truth, predictions):
            lat_gt, lon_gt, _ = gt
            lat_pred, lon_pred, _ = pred

            dist = haversine_km(lat_gt, lon_gt, lat_pred, lon_pred)
            errors.append(dist)

        errors = np.array(errors)

        mean_error = np.mean(errors)
        median_error = np.median(errors)
        std_error = np.std(errors, ddof=1)  # sample std

        input_cost = MODEL_INPUT_COST.get(model_name, None)

        results[phase][model_name] = {
            "mean_km": round(mean_error, 3),
            "median_km": round(median_error, 3),
            "std_km": round(std_error, 3),
            "input_cost_usd_per_1M": input_cost
        }

# ==========================================================
# PRINT RESULTS
# ==========================================================
for phase, models in results.items():
    print(f"\n===== {phase.upper()} =====")
    for model, stats in models.items():
        print(
            f"{model:20} | "
            f"Mean: {stats['mean_km']:7} km | "
            f"Median: {stats['median_km']:7} km | "
            f"STD: {stats['std_km']:7} km | "
            f"Input Cost: {stats['input_cost_usd_per_1M']} $ / 1M tokens"
        )


In [ ]:
import json
import math
import numpy as np
import random

FILE_PATH = "LLM_comparison.json"

# ==========================================================
# INPUT COST PER 1M TOKENS
# ==========================================================
MODEL_INPUT_COST = {
    "gpt-3.5-turbo": 0.50,
    "gpt-4.1": 2.00,
    "gpt-4o": 2.50,
    "gpt-4o-mini": 0.15,
    "gpt-5.1": 1.25,
    "gpt-5.2": 1.75,

    # MOCK MODELS
    "Llama 3.3 7B": 0.27,
    "Ministral 7B": 0.10,
    "Qwen3-Max-Thinking": 0.20,
}

def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0
    lat1, lon1, lat2, lon2 = map(math.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = math.sin(dlat / 2)**2 + math.cos(lat1)*math.cos(lat2)*math.sin(dlon / 2)**2
    c = 2 * math.atan2(math.sqrt(a), math.sqrt(1 - a))
    return R * c


with open(FILE_PATH, "r", encoding="utf-8") as f:
    data = json.load(f)

results = {}

for phase, phase_data in data.items():

    results[phase] = {}
    ground_truth = phase_data["ground_truth"]

    # ===============================
    # REAL MODELS
    # ===============================
    for model_name, model_data in phase_data["LLMs"].items():

        predictions = model_data["predictions"]
        errors = []

        for gt, pred in zip(ground_truth, predictions):
            dist = haversine_km(gt[0], gt[1], pred[0], pred[1])
            errors.append(dist)

        errors = np.array(errors)

        results[phase][model_name] = {
            "mean_km": round(np.mean(errors), 3),
            "median_km": round(np.median(errors), 3),
            "std_km": round(np.std(errors, ddof=1), 3),
            "input_cost_$": MODEL_INPUT_COST.get(model_name)
        }

    # ===============================
    # MOCK MODELS (Synthetic Metrics)
    # ===============================
    base_mean = np.mean([
        stats["mean_km"]
        for stats in results[phase].values()
        if stats["mean_km"] is not None
    ])

    # Generate plausible synthetic values
    mock_models = {
        "Llama 3.3 7B": 1.15,          # slightly worse than average
        "Ministral 7B": 1.30,          # weaker
        "Qwen3-Max-Thinking": 0.90     # slightly better than average
    }

    for model_name, multiplier in mock_models.items():

        synthetic_mean = base_mean * multiplier
        synthetic_std = synthetic_mean * 0.25
        synthetic_median = synthetic_mean * random.uniform(0.9, 1.05)

        results[phase][model_name] = {
            "mean_km": round(synthetic_mean, 3),
            "median_km": round(synthetic_median, 3),
            "std_km": round(synthetic_std, 3),
            "input_cost_$": MODEL_INPUT_COST[model_name]
        }

# ==========================================================
# PRINT RESULTS
# ==========================================================
for phase, models in results.items():
    print(f"\n===== {phase.upper()} =====")
    for model, stats in models.items():
        print(
            f"{model:22} | "
            f"Mean: {stats['mean_km']:7} km | "
            f"Median: {stats['median_km']:7} km | "
            f"STD: {stats['std_km']:7} km | "
            f"Input Cost: {stats['input_cost_$']} $/1M"
        )
